# Giai đoạn 3: Xây dựng NLP Baselines

Huấn luyện TF-IDF và Word2Vec trên cột `review_profile` (gợi ý dựa trên review text); nếu thiếu thì fallback sang `movie_profile` để trả về Recommendation.


In [1]:
import json
import math
import random

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Retrieval metrics (same definitions as notebook 05, but used here for real evaluation) ---

def precision_at_k(predicted: list[int], relevant: set[int], k: int) -> float:
    if k <= 0:
        return 0.0
    top_k = predicted[:k]
    if not top_k:
        return 0.0
    hit_count = sum(1 for idx in top_k if idx in relevant)
    return hit_count / k


def recall_at_k(predicted: list[int], relevant: set[int], k: int) -> float:
    if not relevant:
        return 0.0
    top_k = predicted[:k]
    hit_count = sum(1 for idx in top_k if idx in relevant)
    return hit_count / len(relevant)


def ndcg_at_k(predicted: list[int], relevant: set[int], k: int) -> float:
    if k <= 0 or not relevant:
        return 0.0
    top_k = predicted[:k]
    dcg = 0.0
    for rank, idx in enumerate(top_k, start=1):
        if idx in relevant:
            dcg += 1.0 / math.log2(rank + 1)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(r + 1) for r in range(1, ideal_hits + 1))
    return (dcg / idcg) if idcg > 0 else 0.0


def build_genre_relevance(df: pd.DataFrame, min_overlap: int = 1) -> dict[int, set[int]]:
    """Weak supervision: movies are relevant if they share >= min_overlap genres."""
    # Expect df['genres'] as comma-separated string.
    genres_list = []
    for g in df["genres"].fillna(""):
        parts = [p.strip().lower() for p in str(g).split(",") if p.strip()]
        genres_list.append(set(parts))

    gold: dict[int, set[int]] = {}
    for i, gi in enumerate(genres_list):
        if not gi:
            gold[i] = set()
            continue
        rel = set()
        for j, gj in enumerate(genres_list):
            if i == j:
                continue
            if len(gi.intersection(gj)) >= min_overlap:
                rel.add(j)
        gold[i] = rel
    return gold


def split_query_candidate(n: int, query_ratio: float = 0.2, seed: int = 42) -> tuple[list[int], list[int]]:
    rng = random.Random(seed)
    indices = list(range(n))
    rng.shuffle(indices)
    qn = max(1, int(round(n * query_ratio)))
    query = indices[:qn]
    cand = indices[qn:]
    if not cand:
        # fallback: ensure candidate is non-empty
        cand = query[-1:]
        query = query[:-1]
    return query, cand


def topk_from_similarity(sim_row: np.ndarray, exclude_self: int, candidates: list[int], top_k: int) -> list[int]:
    scores = [(j, float(sim_row[j])) for j in candidates if j != exclude_self]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [j for j, s in scores[:top_k]]


def evaluate_model(sim: np.ndarray, query_ids: list[int], candidate_ids: list[int], gold: dict[int, set[int]], k_values=(5, 10)) -> dict:
    out = {}
    for k in k_values:
        ps, rs, ns = [], [], []
        tp = fp = fn = tn = 0
        for q in query_ids:
            pred = topk_from_similarity(sim[q], q, candidate_ids, top_k=k)
            rel = gold.get(q, set())
            # metrics
            ps.append(precision_at_k(pred, rel, k))
            rs.append(recall_at_k(pred, rel, k))
            ns.append(ndcg_at_k(pred, rel, k))
            # confusion-like counts
            pred_set = set(pred)
            # define "universe" as candidates
            for c in candidate_ids:
                if c == q:
                    continue
                is_rel = c in rel
                is_pred = c in pred_set
                if is_pred and is_rel:
                    tp += 1
                elif is_pred and not is_rel:
                    fp += 1
                elif (not is_pred) and is_rel:
                    fn += 1
                else:
                    tn += 1
        out[f"precision@{k}"] = float(np.mean(ps)) if ps else 0.0
        out[f"recall@{k}"] = float(np.mean(rs)) if rs else 0.0
        out[f"ndcg@{k}"] = float(np.mean(ns)) if ns else 0.0
        out[f"confusion@{k}"] = {"tp": tp, "fp": fp, "fn": fn, "tn": tn}
    return out


try:
    df = pd.read_csv("cleaned_profiles.csv").fillna("")
    print(f"Đã tải {len(df)} phim từ 'cleaned_profiles.csv'\n")

    # We focus on the thesis statement: recommend movies based on review text.
    # Use review_profile if present; fallback to movie_profile.
    text_col = "review_profile" if "review_profile" in df.columns else "movie_profile"
    print(f"Sử dụng cột text: {text_col}")

    # Build weak relevance labels using shared genre overlap.
    gold = build_genre_relevance(df, min_overlap=1)

    # Query/candidate split
    query_ids, candidate_ids = split_query_candidate(len(df), query_ratio=0.2, seed=42)
    print(f"Split: query={len(query_ids)} movies, candidate={len(candidate_ids)} movies")

    # --- Grid configs ---
    k_values = (5, 10)
    configs = []

    # TF-IDF configs
    tfidf_grid = [
        {"name": "tfidf_unigram", "ngram_range": (1, 1), "min_df": 1, "max_df": 1.0},
        {"name": "tfidf_uni_bigram", "ngram_range": (1, 2), "min_df": 1, "max_df": 1.0},
        {"name": "tfidf_uni_bigram_min2", "ngram_range": (1, 2), "min_df": 2, "max_df": 1.0},
    ]

    # Word2Vec configs (if gensim available)
    w2v_grid = [
        {"name": "w2v_dim50", "vector_size": 50, "window": 5, "min_count": 1, "sg": 1},
        {"name": "w2v_dim100", "vector_size": 100, "window": 5, "min_count": 1, "sg": 1},
    ]

    # --- 1) TF-IDF ---
    print("\n=== TF-IDF (grid) ===")
    for cfg in tfidf_grid:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=cfg["ngram_range"],
            min_df=cfg["min_df"],
            max_df=cfg["max_df"],
        )
        X = vectorizer.fit_transform(df[text_col].astype(str).tolist())
        sim = cosine_similarity(X, X)
        metrics = evaluate_model(sim, query_ids, candidate_ids, gold, k_values=k_values)
        row = {"model": cfg["name"], **{k: v for k, v in metrics.items() if not k.startswith("confusion")}}
        configs.append({"model": cfg["name"], "type": "tfidf", "params": cfg, "metrics": metrics})
        print(cfg["name"], json.dumps(row, ensure_ascii=False, indent=2))

    # --- 2) Word2Vec ---
    print("\n=== Word2Vec (grid) ===")
    try:
        from gensim.models import Word2Vec

        # Simple tokenization: split by whitespace (text already cleaned in notebook 02)
        tokenized = [str(t).split() for t in df[text_col].astype(str).tolist()]

        def average_embedding(tokens: list[str], model: Word2Vec, dim: int) -> np.ndarray:
            vecs = [model.wv[t] for t in tokens if t in model.wv]
            if not vecs:
                return np.zeros(dim, dtype=np.float32)
            return np.mean(vecs, axis=0)

        for cfg in w2v_grid:
            model = Word2Vec(
                sentences=tokenized,
                vector_size=cfg["vector_size"],
                window=cfg["window"],
                min_count=cfg["min_count"],
                workers=4,
                sg=cfg["sg"],
                seed=42,
            )
            emb = np.vstack([average_embedding(toks, model, dim=cfg["vector_size"]) for toks in tokenized])
            sim = cosine_similarity(emb, emb)
            metrics = evaluate_model(sim, query_ids, candidate_ids, gold, k_values=k_values)
            row = {"model": cfg["name"], **{k: v for k, v in metrics.items() if not k.startswith("confusion")}}
            configs.append({"model": cfg["name"], "type": "word2vec", "params": cfg, "metrics": metrics})
            print(cfg["name"], json.dumps(row, ensure_ascii=False, indent=2))

    except ImportError:
        print("gensim chưa được cài. Bỏ qua Word2Vec grid. Chạy: pip install gensim")

    # --- Summary table ---
    rows = []
    for c in configs:
        m = c["metrics"]
        rows.append(
            {
                "model": c["model"],
                "precision@5": m.get("precision@5", 0.0),
                "recall@5": m.get("recall@5", 0.0),
                "ndcg@5": m.get("ndcg@5", 0.0),
                "precision@10": m.get("precision@10", 0.0),
                "recall@10": m.get("recall@10", 0.0),
                "ndcg@10": m.get("ndcg@10", 0.0),
            }
        )

    summary = pd.DataFrame(rows).sort_values(by=["ndcg@10", "ndcg@5"], ascending=False)
    display(summary)

    # Save metrics for notebook 05
    with open("eval_results.json", "w", encoding="utf-8") as f:
        json.dump(configs, f, ensure_ascii=False, indent=2)
    print("\nĐã lưu eval_results.json để notebook 05 dùng lại.")

    # =========================================================
    # Multi-field Search + Weighted Re-ranking (Demo)
    # =========================================================
    # Hợp nhất 3 tín hiệu:
    # - Title lexical score (tên riêng / gõ gần đúng)
    # - Genre keyword score (thể loại)
    # - Semantic score (TF-IDF query -> profile)
    # Sau đó weighted fusion theo độ dài query.

    import re
    from difflib import SequenceMatcher

    def _norm_tokens(s: str) -> list[str]:
        return [t for t in re.sub(r"[^a-zA-Z0-9\s]+", " ", str(s).lower()).split() if t]

    def title_lexical_score(query: str, title: str) -> float:
        q = " ".join(_norm_tokens(query))
        t = " ".join(_norm_tokens(title))
        if not q or not t:
            return 0.0
        ratio = SequenceMatcher(None, q, t).ratio()  # 0..1
        qset = set(q.split())
        tset = set(t.split())
        overlap = (len(qset & tset) / max(1, len(qset))) if qset else 0.0
        return 0.6 * ratio + 0.4 * overlap

    def genre_keyword_score(query: str, genres_str: str) -> float:
        q_tokens = set(_norm_tokens(query))
        if not q_tokens:
            return 0.0
        genres = {g.strip().lower() for g in str(genres_str).split(",") if g and g.strip()}
        if not genres:
            return 0.0
        genre_tokens = set()
        for g in genres:
            genre_tokens.update(_norm_tokens(g))
        hits = sum(1 for tok in q_tokens if tok in genre_tokens)
        return hits / max(1, len(q_tokens))

    def choose_weights(query: str) -> tuple[float, float, float]:
        n = len(_norm_tokens(query))
        if n <= 3:
            return (1.0, 0.8, 0.3)
        if n <= 6:
            return (0.7, 0.8, 0.6)
        return (0.2, 0.8, 1.0)

    # Semantic index (TF-IDF) trên movie_profile để bắt được overview+genres+review snippets.
    semantic_col = "movie_profile" if "movie_profile" in df.columns else text_col
    semantic_vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
    )
    semantic_X = semantic_vectorizer.fit_transform(df[semantic_col].astype(str).tolist())

    def semantic_score(query: str) -> np.ndarray:
        qv = semantic_vectorizer.transform([str(query)])
        sims = cosine_similarity(qv, semantic_X).ravel()
        if sims.size == 0:
            return sims
        mx = float(sims.max())
        if mx <= 0:
            return sims
        return sims / mx

    def smart_search(query: str, top_k: int = 10) -> pd.DataFrame:
        w_title, w_genre, w_sem = choose_weights(query)
        sem = semantic_score(query)

        title_scores = df["title"].apply(lambda t: title_lexical_score(query, t)).to_numpy(dtype=float)
        genre_scores = df["genres"].apply(lambda g: genre_keyword_score(query, g)).to_numpy(dtype=float)

        final = (w_title * title_scores) + (w_genre * genre_scores) + (w_sem * sem)

        out = df[["tmdb_id", "title", "genres"]].copy()
        out["score_title"] = title_scores
        out["score_genre"] = genre_scores
        out["score_semantic"] = sem
        out["score_final"] = final
        out = out.sort_values("score_final", ascending=False).head(top_k)
        out["weights"] = f"title={w_title}, genre={w_genre}, semantic={w_sem}"
        return out

    print("\n=== Demo: Multi-field Search + Weighted Re-rank ===")
    demo_queries = [
        "Inception",
        "action horror",
        "mind-bending dream and time loop",
    ]
    for q in demo_queries:
        print(f"\nQuery: {q}")
        display(smart_search(q, top_k=8))

except FileNotFoundError:
    print("Lỗi: Không tìm thấy 'cleaned_profiles.csv'. Xin hãy chạy notebook số 02 trước.")


Đã tải 4900 phim từ 'cleaned_profiles.csv'

Sử dụng cột text: review_profile
Split: query=980 movies, candidate=3920 movies

=== TF-IDF (grid) ===
tfidf_unigram {
  "model": "tfidf_unigram",
  "precision@5": 0.5877551020408164,
  "recall@5": 0.0015506204905459846,
  "ndcg@5": 0.6113560768894533,
  "precision@10": 0.5348979591836736,
  "recall@10": 0.0029901754047758316,
  "ndcg@10": 0.5642448341712255
}
tfidf_uni_bigram {
  "model": "tfidf_uni_bigram",
  "precision@5": 0.586122448979592,
  "recall@5": 0.0015468143184411498,
  "ndcg@5": 0.6102968310584868,
  "precision@10": 0.5303061224489797,
  "recall@10": 0.002979345015537566,
  "ndcg@10": 0.5612598435281245
}
tfidf_uni_bigram_min2 {
  "model": "tfidf_uni_bigram_min2",
  "precision@5": 0.5895918367346938,
  "recall@5": 0.001553416776498262,
  "ndcg@5": 0.6145552838340501,
  "precision@10": 0.5395918367346938,
  "recall@10": 0.003029515639768678,
  "ndcg@10": 0.5691830525824297
}

=== Word2Vec (grid) ===


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

w2v_dim50 {
  "model": "w2v_dim50",
  "precision@5": 0.5530612244897959,
  "recall@5": 0.001388048186409703,
  "ndcg@5": 0.5747365428424648,
  "precision@10": 0.49918367346938775,
  "recall@10": 0.0026976978913814927,
  "ndcg@10": 0.5278413485120275
}


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


w2v_dim100 {
  "model": "w2v_dim100",
  "precision@5": 0.5606122448979591,
  "recall@5": 0.0014107288047842097,
  "ndcg@5": 0.5835991354361132,
  "precision@10": 0.5074489795918367,
  "recall@10": 0.0027490209583329165,
  "ndcg@10": 0.5366964721463052
}


...

,tmdb_id,title,genres,score_title,score_genre,score_semantic,score_final,weights
4701,27205,Inception,"Action, Adventure, Science Fiction",1.000000,0.0,1.0,1.300000,"title=1.0, genre=0.8, semantic=0.3"
2123,1144122,Extinction,Horror,0.442105,0.0,0.0,0.442105,"title=1.0, genre=0.8, semantic=0.3"
2983,429415,Extinction,"Action, Drama, Science Fiction, Thriller",0.442105,0.0,0.0,0.442105,"title=1.0, genre=0.8, semantic=0.3"
2214,72976,Lincoln,"Drama, History",0.375000,0.0,0.0,0.375000,"title=1.0, genre=0.8, semantic=0.3"
2786,211672,Minions,"Adventure, Animation, Comedy, Family",0.375000,0.0,0.0,0.375000,"title=1.0, genre=0.8, semantic=0.3"
2420,993710,Back in Action,"Action, Comedy",0.365217,0.0,0.0,0.365217,"title=1.0, genre=0.8, semantic=0.3"
2781,52150,Zincirbozan,"Crime, Drama, History",0.360000,0.0,0.0,0.360000,"title=1.0, genre=0.8, semantic=0.3"
3334,653460,Corporation,"Crime, Thriller",0.360000,0.0,0.0,0.360000,"title=1.0, genre=0.8, semantic=0.3"



Query: action horror


,tmdb_id,title,genres,score_title,score_genre,score_semantic,score_final,weights
1347,1275882,Sakristan mayor,"Action, Horror",0.257143,1.0,1.000000,1.357143,"title=1.0, genre=0.8, semantic=0.3"
4572,1205225,Gates of Flesh,"Action, Horror, Thriller",0.222222,1.0,0.442916,1.155097,"title=1.0, genre=0.8, semantic=0.3"
3369,1992,Planet Terror,"Action, Horror, Thriller",0.323077,1.0,0.101048,1.153391,"title=1.0, genre=0.8, semantic=0.3"
3889,1244531,Beast of War,"Action, Horror, Thriller, War",0.240000,1.0,0.336398,1.140920,"title=1.0, genre=0.8, semantic=0.3"
4132,330070,Terra Formars,"Action, Horror, Science Fiction",0.230769,1.0,0.227430,1.098998,"title=1.0, genre=0.8, semantic=0.3"
599,84635,Vampire Cop,"Action, Horror",0.150000,1.0,0.452387,1.085716,"title=1.0, genre=0.8, semantic=0.3"
1320,291540,Why Horror?,"Documentary, Horror",0.565217,0.5,0.400126,1.085255,"title=1.0, genre=0.8, semantic=0.3"
4709,1084222,Operation Blood Hunt,"Action, Adventure, Horror",0.254545,1.0,0.094766,1.082975,"title=1.0, genre=0.8, semantic=0.3"



Query: mind-bending dream and time loop


,tmdb_id,title,genres,score_title,score_genre,score_semantic,score_final,weights
4701,27205,Inception,"Action, Adventure, Science Fiction",0.175610,0.0,0.939212,0.686454,"title=0.7, genre=0.8, semantic=0.6"
4094,587792,Palm Springs,"Comedy, Romance, Science Fiction",0.136364,0.0,0.981051,0.684085,"title=0.7, genre=0.8, semantic=0.6"
1770,4977,Paprika,"Animation, Science Fiction, Thriller",0.061538,0.0,1.000000,0.643077,"title=0.7, genre=0.8, semantic=0.6"
3686,434853,Space/Time,"Action, Science Fiction, Thriller",0.238095,0.0,0.656836,0.560768,"title=0.7, genre=0.8, semantic=0.6"
1423,468272,Loop,"Drama, Mystery",0.200000,0.0,0.686573,0.551944,"title=0.7, genre=0.8, semantic=0.6"
485,220289,Coherence,"Science Fiction, Thriller",0.087805,0.0,0.736353,0.503275,"title=0.7, genre=0.8, semantic=0.6"
2707,1385536,Sore: A Wife from the Future,"Drama, Fantasy, Romance, Science Fiction",0.162712,0.0,0.622053,0.487130,"title=0.7, genre=0.8, semantic=0.6"
2256,137113,Edge of Tomorrow,"Action, Science Fiction",0.175000,0.0,0.555580,0.455848,"title=0.7, genre=0.8, semantic=0.6"
